In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load main files
pdc_switchers_csv = pd.read_csv(r'C:\Users\laiar\Desktop\Classe IDIBAPS\data_sintetica\Data-sintetica\Data-sintetica\src\csv merged\pdc_monthly_wide_clusters.csv')
switchers_1517 = pdc_switchers_csv['idCas'].unique()

# Load dispensing tables
df_23_dispensats = pd.read_csv(r"C:\Users\laiar\Desktop\DATOS_PADRIS_nets\23_antipsicotics_dispensats.csv", low_memory=False)
df_25_mhda = pd.read_csv(r"C:\Users\laiar\Desktop\DATOS_PADRIS_nets\25_taula_farmacs_mhda_2010_2019_Final.csv", low_memory=False)

# LAI intervals lookup
LAI_INTERVALS = {
    'ABILIFY MAINTENA 400MG 1 VIAL POLVO +1 VIAL DISOLV SUSPENS LIBER PROLONG': 28,
    'ABILIFY MAINTENA 300MG 1 VIAL POLVO +1 VIAL DISOLV SUSPENS LIBER PROLONG': 28,
    'TREVICTA 175MG 1 JERINGA PREC 0,875ML SUSP INYECT LIBERACION PROL': 84,
    'TREVICTA 350MG 1 JERINGA PRECARGADA 1,750ML SUSP INY LIBER PROLONG': 84,
    'TREVICTA 525MG 1 JERINGA PRECARGADA 2,625ML SUSP INY LIBER PROLONG': 84,
    'TREVICTA 263MG 1 JERINGA PREC 1,315ML + 2 AGUJAS SUSP INY LIBER PROL': 84,
    'XEPLION 50MG 1 JERINGA PRECARG 0,5ML SUSPEN INYEC LIBERAC PROLONG': 28,
    'XEPLION 75MG 1 JERINGA PRECARG 0,75ML SUSPEN INYEC LIBERAC PROLONG': 28,
    'XEPLION 100MG 1 JERINGA PRECARG 1ML SUSPENS INYEC LIBERAC PROLONG': 28,
    'XEPLION 150MG 1 JERINGA PRECARG 1,5ML SUSPEN INYEC LIBERAC PROLONG': 28,
    'XEPLION 50MG 1 JER PREC + 2 AGU SUSPENS INYECT LIBERACION PROLONGADA': 28,
    'XEPLION 75MG 1 JER PREC + 2 AGU SUSPENS INYECT LIBERACION PROLONGADA': 28,
    'XEPLION 100MG 1 JER PREC + 2 AGU SUSPENS INYECT LIBERACION PROLONGADA': 28,
    'XEPLION 150MG 1 JER PREC + 2 AGU SUSPENS INYECT LIBERACION PROLONGADA': 28,
    'RISPERDAL CONSTA 25MG/VIAL 1 VIAL + 1 JER PRECARG': 14,
    'RISPERDAL CONSTA 37,5MG/VIAL 1 VIAL + 1 JER PRECAR': 14,
    'RISPERDAL CONSTA 50MG/VIAL 1 VIAL + 1 JER PRECARG': 14,
    'ZYPADHERA 210MG 1 VIAL POLVO + 1 VIAL DISOLV SUSP INY LIBERAC PROLONG': 14,
    'ZYPADHERA 300MG 1 VIAL POLVO + 1 VIAL DISOLV SUSP INY LIBERAC PROLONG': 28,
    'ZYPADHERA 405MG 1 VIAL POLVO + 1 VIAL DISOLV SUSP INY LIBERAC PROLONG': 28,
    'HALDOL DECANOATE 100 mg/ml 1 Solución inyectable': 28,
    'HALDOL DEPOT 5 AMP. 100 MG.': 28,
}

IMMEDIATE_ACTION = [
    'HALOPERIDOL ESTEVE 5MG/ML 5 AMPOLLAS INYECTABLE',
    'DOGMATIL 50MG/ML 12 AMPOLLAS 2ML SOLUCION INYECTABLE',
    'DOGMATIL 50MG/ML SOLUCION INYECTABLE 12 AMPOLLAS DE 2ML',
    'DOGMATIL 50 mg/ml  5 Soluc inyect/CSE013078',
    'ZYPREXA 10MG 1 VIAL POLVO PARA SOLUCION INYECTABLE',
    'ABILIFY 7,5MG/ML 1 VIAL 1,3ML SOLUCION INYECTABLE',
]

def classify_product(product_name):
    if product_name in LAI_INTERVALS:
        return 'LAI'
    elif product_name in IMMEDIATE_ACTION:
        return 'IMMEDIATE_ACTION'
    else:
        return 'ORAL'

def anymes_to_date(anymes):
    anymes = str(int(anymes))
    return pd.Timestamp(year=int(anymes[:4]), month=int(anymes[4:]), day=15)

# Apply classification and dates
df_23_dispensats['formulation_type'] = df_23_dispensats['DES_HIS_Producte'].apply(classify_product)
df_23_dispensats['dispensing_date'] = df_23_dispensats['Any_Mes'].apply(anymes_to_date)
df_25_mhda['formulation_type'] = df_25_mhda['(HIS) Producte Desc'].apply(classify_product)
df_25_mhda['dispensing_date'] = df_25_mhda['Any_Mes'].apply(anymes_to_date)

print("Tables loaded and classified.")
print(df_23_dispensats['formulation_type'].value_counts())
print(df_25_mhda['formulation_type'].value_counts())

Tables loaded and classified.
formulation_type
ORAL                6731903
LAI                 1398094
IMMEDIATE_ACTION      10472
Name: count, dtype: int64
formulation_type
ORAL                835258
LAI                   2556
IMMEDIATE_ACTION        12
Name: count, dtype: int64


In [3]:
# Generation classification
first_gen_atc = ['N05AD01', 'N05AH06', 'N05AL01', 'N05AL05']
second_gen_atc = ['N05AH02', 'N05AH03', 'N05AH04', 'N05AX08', 'N05AX12', 'N05AX13', 'N05AE05']

first_gen_lai = ['HALDOL DECANOATE 100 mg/ml 1 Solución inyectable', 'HALDOL DEPOT 5 AMP. 100 MG.']
second_gen_lai = [k for k in LAI_INTERVALS.keys() if k not in first_gen_lai]

# Add generation to dispensing tables
df_23_dispensats['generation'] = df_23_dispensats['ID_HIS_Subgrup_7_ATC'].apply(
    lambda x: '1st generation' if x in first_gen_atc else ('2nd generation' if x in second_gen_atc else 'unknown')
)
df_25_mhda['generation'] = df_25_mhda['(HIS) Producte Desc'].apply(
    lambda x: '1st generation' if x in first_gen_lai else '2nd generation'
)

# Merge generation into pdc_switchers_csv via dispensing tables
# Pre-switch: use oral dispensings
pre_gen = df_23_dispensats[
    (df_23_dispensats['idCas'].isin(switchers_1517)) &
    (df_23_dispensats['formulation_type'] == 'ORAL')
].groupby('idCas')['generation'].agg(lambda x: x.mode()[0]).reset_index()
pre_gen.columns = ['idCas', 'pre_generation']

# Post-switch: use LAI dispensings
post_gen = pd.concat([
    df_23_dispensats[(df_23_dispensats['idCas'].isin(switchers_1517)) & 
                     (df_23_dispensats['formulation_type'] == 'LAI')][['idCas', 'generation']],
    df_25_mhda[df_25_mhda['idCas'].isin(switchers_1517)][['idCas', 'generation']]
]).groupby('idCas')['generation'].agg(lambda x: x.mode()[0]).reset_index()
post_gen.columns = ['idCas', 'post_generation']

pdc_gen = pdc_switchers_csv.merge(pre_gen, on='idCas', how='left')
pdc_gen = pdc_gen.merge(post_gen, on='idCas', how='left')

print(pdc_gen['pre_generation'].value_counts())
print(pdc_gen['post_generation'].value_counts())

pre_generation
2nd generation    995
1st generation     30
Name: count, dtype: int64
post_generation
2nd generation    1025
Name: count, dtype: int64
